# 02 - Modelagem, baselines e forecast

Este notebook explica passo a passo como o projeto transforma o histórico real em previsões comparáveis.

A estratégia do projeto é deliberadamente interpretável:

- construir baselines simples;
- comparar todos por backtest;
- selecionar o modelo com menor WAPE;
- usar o modelo selecionado no dashboard executivo.

Isso cria uma base sólida para portfólio e para evolução futura com modelos mais sofisticados.

## 1. Preparação

Vamos reutilizar as funções do pacote `revenue_forecast_dashboard`, que são as mesmas usadas pelo dashboard e pelo script de avaliação.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from revenue_forecast_dashboard.data import SCENARIOS, load_history  # noqa: E402
from revenue_forecast_dashboard.model import (  # noqa: E402
    build_model_registry,
    evaluate_predictions,
)

PROJECT_ROOT

## 2. Carregar histórico no formato do projeto

A função `load_history()` já aplica os tratamentos usados pelo dashboard:

- lê `train.csv`;
- junta metadados de `stores.csv`;
- cria `store`, `region`, `revenue` e `promo`;
- ordena os dados por data, loja e família.

Neste projeto, `revenue` representa o volume `sales`, não receita monetária.

In [ ]:
history = load_history()
history.head()

In [ ]:
pd.Series(
    {
        "linhas": len(history),
        "data_inicial": history["date"].min(),
        "data_final": history["date"].max(),
        "lojas": history["store_nbr"].nunique(),
        "familias": history["family"].nunique(),
        "series_loja_familia": history[["store_nbr", "family"]].drop_duplicates().shape[0],
    }
)

## 3. Definir janela de validação

Para simular uma situação real, separamos os últimos dias do histórico como validação.

O modelo só enxerga o período de treino. Depois ele prevê a janela de validação e comparamos previsão contra o valor real.

O script oficial usa `16` dias porque essa é a janela final disponível na competição para validação local do projeto.

In [ ]:
VALIDATION_DAYS = 16

latest_date = history["date"].max()
cutoff_date = latest_date - pd.Timedelta(VALIDATION_DAYS, unit="D")

train = history[history["date"] <= cutoff_date].copy()
validation = history[history["date"] > cutoff_date].copy()
validation_dates = pd.date_range(cutoff_date + pd.Timedelta(1, unit="D"), periods=VALIDATION_DAYS)

pd.Series(
    {
        "treino_ate": cutoff_date,
        "validacao_inicio": validation_dates.min(),
        "validacao_fim": validation_dates.max(),
        "linhas_treino": len(train),
        "linhas_validacao": len(validation),
    }
)

## 4. Promoções futuras durante o backtest

No backtest, usamos a própria janela de validação para simular o conhecimento das promoções futuras. Isso replica o caso em que a empresa já possui calendário promocional planejado para os próximos dias.

In [ ]:
future_promotions = {
    (pd.Timestamp(row.date), int(row.store_nbr), str(row.family)): float(row.onpromotion)
    for row in validation.itertuples(index=False)
}

len(future_promotions)

## 5. Modelos comparados

O projeto compara cinco modelos interpretáveis:

1. `LastValueBaseline`: repete o último valor observado.
2. `MovingAverageBaseline`: usa a média móvel recente.
3. `SeasonalNaiveBaseline`: repete o último valor observado do mesmo dia da semana.
4. `WeekdayAverageBaseline`: usa a média histórica recente por dia da semana.
5. `SeasonalPromotionTrendForecaster`: combina sazonalidade semanal, média recente, tendência e promoção.

A comparação evita escolher um modelo pela aparência. O critério final é quantitativo.

In [ ]:
models = build_model_registry(max_forecast_days=VALIDATION_DAYS)
pd.DataFrame(
    {
        "modelo": [model.name for model in models],
        "tipo": ["baseline" if "Baseline" in model.name else "candidate" for model in models],
        "horizonte_maximo": [model.max_forecast_days for model in models],
        "lookback_dias": [model.lookback_days for model in models],
    }
)

## 6. Gerar previsão para um modelo

Cada modelo retorna previsões em três cenários:

- `conservador`
- `esperado`
- `otimista`

Para avaliação quantitativa, usamos o cenário `esperado`.

In [ ]:
example_model = models[3]
example_forecast = example_model.predict(
    train,
    future_promotions,
    SCENARIOS,
    forecast_dates=validation_dates,
)

example_forecast.head()

In [ ]:
example_forecast.groupby("scenario").agg(
    linhas=("revenue", "size"),
    vendas_previstas=("revenue", "sum"),
    confianca_media=("confidence", "mean"),
    risco_medio=("risk_score", "mean"),
)

## 7. Comparar previsão contra real

A avaliação junta `validation` com a previsão pelo trio:

`date + store_nbr + family`

Essa junção garante que cada valor previsto seja comparado com a venda real da mesma frente comercial e data.

In [ ]:
expected = example_forecast[example_forecast["scenario"] == "esperado"][
    ["date", "store_nbr", "family", "revenue"]
].rename(columns={"revenue": "prediction"})

scored = validation.merge(expected, on=["date", "store_nbr", "family"], how="inner")
scored[["date", "store", "family", "revenue", "prediction"]].head()

In [ ]:
evaluate_predictions(scored["revenue"], scored["prediction"])

## 8. Rodar comparação completa dos modelos

A célula abaixo reproduz a lógica central de `scripts/evaluate_model.py`, mas em formato exploratório para estudo.

In [ ]:
comparison_rows = []
scored_by_model = {}

for model in build_model_registry(max_forecast_days=VALIDATION_DAYS):
    forecast = model.predict(
        train,
        future_promotions,
        SCENARIOS,
        forecast_dates=validation_dates,
    )
    expected = forecast[forecast["scenario"] == "esperado"][
        ["date", "store_nbr", "family", "revenue"]
    ].rename(columns={"revenue": "prediction"})
    scored_model = validation.merge(expected, on=["date", "store_nbr", "family"], how="inner")
    scored_by_model[model.name] = scored_model

    metrics = evaluate_predictions(scored_model["revenue"], scored_model["prediction"])
    comparison_rows.append(
        {
            "model": model.name,
            "model_role": "baseline" if "Baseline" in model.name else "candidate",
            "rows_scored": len(scored_model),
            "series_scored": scored_model[["store_nbr", "family"]].drop_duplicates().shape[0],
            **metrics,
        }
    )

comparison = pd.DataFrame(comparison_rows).sort_values(["wape", "smape", "rmse"])
comparison

In [ ]:
fig = px.bar(
    comparison.sort_values("wape", ascending=False),
    x="wape",
    y="model",
    color="model_role",
    orientation="h",
    title="Comparação de modelos por WAPE",
    labels={"wape": "WAPE", "model": "modelo", "model_role": "tipo"},
)
fig.update_xaxes(tickformat=".1%")
fig.show()

## 9. Análise do modelo vencedor

Depois de ordenar por WAPE, o primeiro modelo da tabela é o vencedor da janela de validação.

Essa decisão é salva em `reports/model_metrics.json` pelo script oficial e lida automaticamente pelo dashboard.

In [ ]:
winner = comparison.iloc[0]
winner

In [ ]:
best_scored = scored_by_model[winner["model"]].copy()
best_scored["error"] = best_scored["prediction"] - best_scored["revenue"]
best_scored["abs_error"] = best_scored["error"].abs()

error_by_family = (
    best_scored.groupby("family", observed=True)
    .agg(
        real=("revenue", "sum"),
        previsto=("prediction", "sum"),
        erro_absoluto=("abs_error", "sum"),
    )
    .assign(wape=lambda df: df["erro_absoluto"] / df["real"])
    .sort_values("wape", ascending=False)
)

error_by_family.head(12)

## 10. Conclusões de modelagem

A abordagem adotada é simples, mas robusta para um projeto executivo:

1. Baselines fortes são indispensáveis. Um modelo novo só faz sentido se superar alternativas simples.
2. WAPE é a métrica principal porque mede erro absoluto ponderado pelo volume total, o que é adequado para planejamento comercial.
3. Métricas complementares como SMAPE, RMSE, RMSLE e Bias ajudam a entender comportamento do erro.
4. O dashboard não usa um modelo fixo no código: ele lê o resultado do benchmark em `reports/model_metrics.json`.
5. A próxima evolução natural seria testar modelos por família, loja ou cluster, além de incorporar feriados, óleo e transações.

Próximo notebook: `03_avaliacao_metricas.ipynb`, com leitura detalhada das métricas finais geradas pelo script reprodutível.